# Create Notebook

In [2]:
import sys
import os

# Add project root
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

from src.hypothesis_tests import (
    run_ttest,
    run_chi_square,
    calculate_margin,
    create_claim_indicator
)

# Load Data

In [3]:
df = pd.read_csv("../data/MachineLearningRating_v3_cleaned.csv")

C:\Users\YENGU\AppData\Local\Temp\ipykernel_7252\1454754216.py:1: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/MachineLearningRating_v3_cleaned.csv")


In [ ]:
# Preview
df.head()

,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [5]:
# Data Preparation 
#  Create marign 
df = calculate_margin(df)

In [6]:
# Create Claim Indicator
df = create_claim_indicator(df)

In [7]:
# Verify
df[["Margin", "HasClaim"]].head()

,Margin,HasClaim
0,21.929825,0
1,21.929825,0
2,0.000000,0
3,512.848070,0
4,0.000000,0


# Hypothesis 1

H₀: There are no risk differences across provinces.
### KPI Selection

Use: Claim Severity (TotalClaims)
Select Provinces

### Example:

-  Gauteng
-  Western Cape

In [8]:
province_a = df[
    df["Province"] == "Gauteng"
]["TotalClaims"]

province_b = df[
    df["Province"] == "Western Cape"
]["TotalClaims"]

# Run T-Test

In [9]:
province_test = run_ttest(
    province_a,
    province_b
)

province_test

{'test': 'Independent T-Test',
 'p_value': np.float64(0.05632044649871941),
 'decision': 'Fail to Reject H0'}

# Interpretation

In [10]:
print(province_test["decision"])
print(province_test["p_value"])

Fail to Reject H0
0.05632044649871941


# Business Interpretation

Example:

In [14]:
if province_test["p_value"] < 0.05:
    print("""
    Provinces exhibit statistically significant
    differences in claim severity.

    This suggests province-specific pricing
    strategies may improve profitability.
    """)

# Hypothesis 2
H₀: There are no risk differences between zip codes.
### KPI Selection
Use: Claim Frequency
### Create Contingency Table

In [12]:
zip_contingency = pd.crosstab(
    df["PostalCode"],
    df["HasClaim"]
)

zip_contingency.head()

HasClaim,0,1
PostalCode,,
1,5329,12
2,1482,6
4,77,0
5,396,4
6,438,2


# Run Chi-Square Test

In [13]:
zip_test = run_chi_square(
    zip_contingency
)

zip_test

{'test': 'Chi-Square',
 'p_value': np.float64(3.152172246339057e-30),
 'decision': 'Reject H0'}

# Business Interpretation

In [15]:
if zip_test["p_value"] < 0.05:
    print("""
    Claim frequency differs significantly
    across zip codes.

    Geographic pricing segmentation may
    reduce underwriting risk.
    """)


    Claim frequency differs significantly
    across zip codes.

    Geographic pricing segmentation may
    reduce underwriting risk.
    


In [19]:
df["PostalCode"].value_counts().head(20)

PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
458      13775
8000     11794
2196     11048
470      10226
7100     10161
1724     10107
4360      9730
302       9531
152       9423
7750      9408
1863      8655
1022      8476
4068      8234
400       6692
4001      6647
Name: count, dtype: int64

# Hypothesis 3
H₀:There is no significant margin difference between zip codes.
### KPI Selection

Use: Margin
### Select Two Zip Codes

In [20]:
zip_a = df[
    df["PostalCode"] == 2000
]["Margin"].dropna()

zip_b = df[
    df["PostalCode"] == 122
]["Margin"].dropna()

In [22]:
df["Margin"] = (
    df["TotalPremium"] - df["TotalClaims"]
)

In [23]:
df["Margin"].describe()

count    1.000098e+06
mean    -2.955694e+00
std      2.367137e+03
min     -3.928486e+05
25%      0.000000e+00
50%      2.157687e+00
75%      2.192982e+01
max      6.528260e+04
Name: Margin, dtype: float64

### Run Welch’s T-Test
-  Why Welch’s T-Test?

Insurance financial data usually contains:

-  skewness
-  unequal variances
-  outliers



In [21]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    zip_a,
    zip_b,
    equal_var=False,
    nan_policy="omit"
)

print("T-statistic:", t_stat)
print("P-value:", p_value)

T-statistic: 1.1639145988804174
P-value: 0.24446241842452013


# Interpretation

-  This means:

There is no statistically significant difference in insurance margin between PostalCode 2000 and PostalCode 122 at the 5% significance level.

### In practical terms:

-  the profitability levels between these two postal codes appear statistically similar 
-  observed differences may be due to random variation rather than meaningful risk differences
-  An independent Welch’s t-test was conducted to evaluate whether average insurance margins differed significantly between PostalCode 2000 and PostalCode 122. 
-  The test produced a p-value of 0.2445, which exceeds the 0.05 significance threshold. 
-  Therefore, the null hypothesis was not rejected, indicating insufficient statistical evidence of a meaningful difference in margin between the two postal code groups.